In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, accuracy_score, classification_report

# Load your dataset
try:
    df = pd.read_csv('/Users/abhi/delay_aware/flight-delay-prediction/data/ot_delaycause1_DL/Airline_Delay_Cause.csv')
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Error: airline_delay.csv not found. Please check the file path.")
    df = pd.DataFrame() # Avoids errors in subsequent cells if file isn't found

Dataset loaded successfully.


In [21]:
if not df.empty:
    # Drop rows with zero flights to avoid division by zero errors
    df = df[df['arr_flights'] > 0].copy()

    # --- Fix for FutureWarning: Handle missing values correctly ---
    # This replaces the old .fillna() line
    df['arr_delay'] = df['arr_delay'].fillna(df['arr_delay'].median())

    # --- TARGET 1: Create a "High Delay Risk" flag for the classifier ---
    df['delay_rate'] = df['arr_del15'] / df['arr_flights']
    df['high_delay_risk'] = (df['delay_rate'] > 0.20).astype(int)

    # --- TARGET 2: Create the AVERAGE delay per flight for the regressor ---
    df['avg_delay_per_flight'] = df['arr_delay'] / df['arr_flights']

    # --- FEATURE PREPARATION ---
    features = ['year', 'month', 'carrier', 'airport', 'arr_flights']
    # Add the two new targets to the modeling dataframe
    df_model = df[features + ['high_delay_risk', 'avg_delay_per_flight']].copy()

    # Encode categorical features ('carrier' and 'airport') into numbers
    for col in ['carrier', 'airport']:
        le = LabelEncoder()
        df_model[col] = le.fit_transform(df_model[col])

    print("Data prepared for modeling. Here are the first 5 rows:")
    print(df_model.head())

Data prepared for modeling. Here are the first 5 rows:
   year  month  carrier  airport  arr_flights  high_delay_risk  \
0  2025      5        0        0         92.0                0   
1  2025      5        0       11         92.0                1   
2  2025      5        0       12        188.0                1   
3  2025      5        0       14         83.0                1   
4  2025      5        0       24       3118.0                1   

   avg_delay_per_flight  
0             19.934783  
1             22.608696  
2             21.978723  
3             23.795181  
4             21.714240  


In [22]:
if not df.empty:
    X = df_model[features]
    y_class = df_model['high_delay_risk']  # Target for Model 1
    # Change this line in Block 3
    y_reg = df['avg_delay_per_flight'] # Use the new, more stable target         # Target for Model 2

    # Split data into 80% for training and 20% for testing
    X_train, X_test, y_class_train, y_class_test, y_reg_train, y_reg_test = train_test_split(
        X, y_class, y_reg, test_size=0.2, random_state=42
    )
    print(f"Data split into {len(X_train)} training samples and {len(X_test)} testing samples.")

Data split into 324089 training samples and 81023 testing samples.


In [23]:
if not df.empty:
    print("\n--- Training Model 1: High Delay Risk Classifier ---")
    
    # Initialize and train the classifier
    classifier = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    classifier.fit(X_train, y_class_train)
    
    # Make predictions and evaluate
    y_class_pred = classifier.predict(X_test)
    
    print("\nClassifier Accuracy:", accuracy_score(y_class_test, y_class_pred))
    print("\nClassification Report:")
    print(classification_report(y_class_test, y_class_pred))


--- Training Model 1: High Delay Risk Classifier ---

Classifier Accuracy: 0.7444305937820125

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.80      0.78     45573
           1       0.72      0.68      0.70     35450

    accuracy                           0.74     81023
   macro avg       0.74      0.74      0.74     81023
weighted avg       0.74      0.74      0.74     81023



In [24]:
if not df.empty:
    print("\n--- Training Model 2: Average Delay Duration Regressor ---")
    
    # Initialize and train the regressor
    regressor = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    regressor.fit(X_train, y_reg_train)
    
    # Make predictions and evaluate
    y_reg_pred = regressor.predict(X_test)
    
    mae = mean_absolute_error(y_reg_test, y_reg_pred)
    
    print(f"\nRegressor Mean Absolute Error (MAE): {mae:.2f} minutes")
    print(f"This means the model's prediction for average delay is off by about {mae:.2f} minutes on average.")


--- Training Model 2: Average Delay Duration Regressor ---

Regressor Mean Absolute Error (MAE): 4.99 minutes
This means the model's prediction for average delay is off by about 4.99 minutes on average.
